In [0]:
import sys
# Clear corrupted matplotlib cache if present
_mpl_keys = [k for k in sys.modules if k.startswith('matplotlib')]
for _k in _mpl_keys:
    del sys.modules[_k]
del _mpl_keys

%matplotlib inline
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, StringType, DateType, TimestampType, DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
import matplotlib.pyplot as plt
from pyspark.ml.stat import Correlation
import pandas as pd
from pyspark.sql.window import Window
import seaborn as sns
from pyspark.sql.functions import col, udf
from pyspark.ml.classification import LogisticRegression
import numpy as np
import matplotlib.patches as mpatches

In [0]:

TABLE_NAME = "dev_gold.pitchvision.injury_risk_input_filtered"

df = spark.table(TABLE_NAME)

print(f"EDA DE TABLA: {TABLE_NAME}")

num_rows = df.count()
num_cols = len(df.columns)

print("\n1. DIMENSIONES")
print(f"Filas: {num_rows:,}")
print(f"Columnas: {num_cols:,}")

print("\n2. ESQUEMA")
df.printSchema()

print("\n3. MUESTRA DE DATOS")
display(df.limit(10))


numeric_cols = [
    field.name for field in df.schema.fields
    if isinstance(field.dataType, NumericType)
]

string_cols = [
    field.name for field in df.schema.fields
    if isinstance(field.dataType, StringType)
]

date_cols = [
    field.name for field in df.schema.fields
    if isinstance(field.dataType, (DateType, TimestampType))
]

other_cols = [
    field.name for field in df.schema.fields
    if field.name not in numeric_cols + string_cols + date_cols
]

print("\n4. TIPOS DE COLUMNAS")
print(f"Columnas numéricas ({len(numeric_cols)}): {numeric_cols}")
print(f"Columnas texto/categóricas ({len(string_cols)}): {string_cols}")
print(f"Columnas fecha/timestamp ({len(date_cols)}): {date_cols}")
print(f"Otras columnas ({len(other_cols)}): {other_cols}")


### Métricas acumuladas

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum as spark_sum

# Definir ventanas para la última semana y el último mes por deportista
window_7d = Window.partitionBy("id_unico_deportista").orderBy("activity_date").rangeBetween(-7*86400, 0)
window_30d = Window.partitionBy("id_unico_deportista").orderBy("activity_date").rangeBetween(-30*86400, 0)

# Convertir activity_date a timestamp si es necesario
df = df.withColumn("activity_date_ts", F.col("activity_date").cast("timestamp"))

# Crear columnas de acumulado
df = df.withColumn(
    "participation_time_minutes_7d_sum",
    F.sum("participation_time_minutes").over(
        Window.partitionBy("id_unico_deportista")
        .orderBy(F.col("activity_date_ts").cast("long"))
        .rangeBetween(-7*86400, 0)
    )
).withColumn(
    "participation_time_minutes_30d_sum",
    F.sum("participation_time_minutes").over(
        Window.partitionBy("id_unico_deportista")
        .orderBy(F.col("activity_date_ts").cast("long"))
        .rangeBetween(-30*86400, 0)
    )
).withColumn(
    "total_distance_7d_sum",
    F.sum("total_distance").over(
        Window.partitionBy("id_unico_deportista")
        .orderBy(F.col("activity_date_ts").cast("long"))
        .rangeBetween(-7*86400, 0)
    )
).withColumn(
    "total_distance_30d_sum",
    F.sum("total_distance").over(
        Window.partitionBy("id_unico_deportista")
        .orderBy(F.col("activity_date_ts").cast("long"))
        .rangeBetween(-30*86400, 0)
    )
)

display(df.select(
    "id_unico_deportista", "activity_date", 
    "participation_time_minutes", "total_distance",
    "participation_time_minutes_7d_sum", "participation_time_minutes_30d_sum",
    "total_distance_7d_sum", "total_distance_30d_sum"
).limit(10))

In [0]:
cols_to_max = [
    "total_distance",
    "max_velocity",
    "max_acceleration",
    "max_deceleration",
    "acc_dcc_distance",
    "acc_dcc_count",
    "acc_count",
    "dcc_count",
    "HSR",
    "sprint_distance",
    "sprint_count"
]

# Calcular solo la desviación respecto al máximo por jugador para cada columna (sin añadir el máximo)
for col_name in cols_to_max:
    max_col_name = f"{col_name}_player_max_tmp"
    dev_col_name = f"{col_name}_dev_player_max"
    window = Window.partitionBy("id_unico_deportista")
    df = df.withColumn(
        max_col_name, F.max(col_name).over(window)
    ).withColumn(
        dev_col_name,
        F.when(
            F.col(max_col_name).isNotNull() & (F.col(max_col_name) != 0),
            (F.col(col_name) - F.col(max_col_name)) / F.col(max_col_name)
        ).otherwise(None)
    ).drop(max_col_name)

display(df.select("id_unico_deportista", *cols_to_max, *[f"{col}_dev_player_max" for col in cols_to_max]).limit(10))

### Borrado de columnas no informativas

In [0]:
columnas_a_eliminar = [
    "session_week",
    "iso_left_repetitions",
    "gps_activity_team_id",
    "session_time_minutes",
    "iso_right_repetitions",
    "nor_left_repetitions",
    "comp_activity_team_id"
]

df = df.drop(*columnas_a_eliminar)

# Preparación de datos

### Configuraciónde variables independientes
Identíficamos qué columns de nuestra tabla limpia van a servir para predecir el riesgo de lesión. Excluimos la variables objetivo (`injury_risk`) y los datos de identidad del jugador y fecha (`id_unico_deportista`, `activity_date`) para que el modelo no se confunda con ellos.
nota: adecuar nombre de la columna variable objetivo y nombre de la tabla

In [0]:
columna_objetivo = "injury_risk"
columnas_excluidas = [columna_objetivo, "id_unico_deportista", "activity_date"]
columnas_features = [c for c in df.columns if c not in columnas_excluidas]
print(f"Número total de variables que procesará el modelo: {len(columnas_features)}")
print("Variables seleccionadas:", columnas_features)

### Eliminar columnas con moda >95%

In [0]:
from pyspark.sql import DataFrame

def eliminar_columnas_moda(df: DataFrame, columnas: list, umbral: float = 0.95):
    cols_a_eliminar = []
    total_filas = df.count()
    for col_name in columnas:
        moda = df.groupBy(col_name).count().orderBy(F.desc("count")).first()
        if moda and moda['count'] / total_filas > umbral:
            cols_a_eliminar.append(col_name)
    print(f"Columnas eliminadas por moda > {umbral*100:.1f}%: {cols_a_eliminar}")
    columnas_restantes = [c for c in columnas if c not in cols_a_eliminar]
    return df.drop(*cols_a_eliminar), columnas_restantes

df_limpio_1, columnas_features = eliminar_columnas_moda(df, columnas_features)
display(df_limpio_1.limit(10))

In [0]:
df_limpio_1 = df_limpio_1.drop("max_deceleration_dev_player_max")

### Eliminar columnas con correlación más del 95%

In [0]:
def eliminar_columnas_covarianza(df: DataFrame, columnas: list, umbral: float = 0.95) -> DataFrame:
    # Solo columnas numéricas
    df_cols = set(df.columns)
    numeric_cols = [c for c in columnas if c in df_cols and isinstance(df.schema[c].dataType, NumericType)]
    assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features_cov", handleInvalid="skip")
    df_vec = assembler.transform(df.select(numeric_cols))
    corr_matrix = Correlation.corr(df_vec, "features_cov", "pearson").head()[0].toArray()
    cols_a_eliminar = set()
    for i in range(len(numeric_cols)):
        for j in range(i+1, len(numeric_cols)):
            if abs(corr_matrix[i, j]) > umbral:
                # No eliminar columnas que contienen '_dev_'
                if not numeric_cols[j].find('_dev_') != -1:
                    cols_a_eliminar.add(numeric_cols[j])
    print(f"Columnas eliminadas por covarianza > {umbral*100:.1f}%: {list(cols_a_eliminar)}")
    return df.drop(*cols_a_eliminar), [c for c in columnas if c not in cols_a_eliminar]

df_limpio_2, columnas_features = eliminar_columnas_covarianza(df_limpio_1, columnas_features, umbral=0.95)
display(df_limpio_2.limit(10))

### Cálculo de pesos

In [0]:
print("Calculando pesos de balanceo para la variable objetivo...")

# Contamos totales de la base de datos actual
total_registros = df_limpio_2.count()
total_positivos = df_limpio_2.filter(F.col(columna_objetivo) == 1).count()
total_negativos = total_registros - total_positivos

# Calculamos los factores de peso
peso_0 = total_registros / (2.0 * total_negativos)
peso_1 = total_registros / (2.0 * total_positivos)

print(f"-> Peso asignado a los días Sanos (0): {peso_0:.4f}")
print(f"-> Peso asignado a los días de Riesgo (1): {peso_1:.4f} (Factor x{peso_1/peso_0:.1f})")

# Añadimos la columna de pesos al DataFrame
df_con_pesos = df_limpio_2.withColumn(
    "weight", 
    F.when(F.col(columna_objetivo) == 1, peso_1).otherwise(peso_0)
)

### Ensamblado de vectores
PySpark requiere que todas las variables independientes estén agrupadas en una única columna de tipo vector. Utilizamos `VectorAssembler` para consolidar todas las columnas seleccionadas en una nueva columna temporal llamada `features_sin_escalar`.

In [0]:
valid_features = [c for c in columnas_features if c in df_con_pesos.columns and df_con_pesos.schema[c].dataType.typeName() != 'timestamp']

assembler = VectorAssembler(
    inputCols=valid_features, 
    outputCol="features_sin_escalar",
    handleInvalid="keep"
)

df_ensamblado = assembler.transform(df_con_pesos)

display(df_ensamblado.select("id_unico_deportista", "activity_date", "features_sin_escalar", "weight",  columna_objetivo).limit(5))

### Escalado de Variables
Como la Regresión Logística utiliza cálculos basados en magnitudes, necesitamos igualar el terreno de juego para que variables con números grandes (ej. Distancia Total en metros) no eclipsen a variables con números pequeños (ej. Ratio ACWR). Ponemos todos los datos en la misma escala (media 0 y desviación estándar 1). Utilizamos `StandardScaler`.

In [0]:
scaler = StandardScaler(
    inputCol="features_sin_escalar", 
    outputCol="features", 
    withStd=True, 
    withMean=True
)

scaler_model = scaler.fit(df_ensamblado)
df_preparado = scaler_model.transform(df_ensamblado)

df_entrenamiento = df_preparado.select("id_unico_deportista", "activity_date", "features", "weight", columna_objetivo)

display(df_entrenamiento.limit(5))

# Configuración y entrenamiento del modelo

### Configuración de la Regresión Logística
Configuramos el algoritmo de Regresión Logística de PySpark. Para proteger el modelo contra el sobreentrenamiento (overfitting) al usar las dos temporadas completas, aplicamos una regularización ligera (`regParam`). Además, especificamos explícitamente el nombre de la columna de entrada de datos (`features`) y la variable objetivo (`{variable objetivo}`).

In [0]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol=columna_objetivo,
    predictionCol="prediction",              # Clasificación estándar 0 o 1 (umbral 50%)
    probabilityCol="probabilidad_vector",   # Vector que contiene la probabilidad de [Sano, Riesgo]
    weightCol="weight",
    regParam=0.01,                          # Penalización suave para evitar overfitting
    threshold=0.5                            # Predice lesión (1) si P(riesgo) >= 0.5
)

print("Algoritmo de Regresión Logística configurado y listo para el entrenamiento.")

### Cross-Validation con GroupKFold por Deportista
En lugar de entrenar con todos los datos y evaluar sobre el mismo conjunto (lo cual produce métricas optimistas), realizamos una validación cruzada agrupada por deportista. De esta forma:
- Ningún deportista aparece simultáneamente en entrenamiento y test (evita fuga de datos).
- Obtenemos métricas realistas de generalización: AUC-ROC, AUC-PR, Precision, Recall y F1.
- Comparamos Regresión Logística vs Random Forest en igualdad de condiciones.

In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import numpy as np

# --- Configuración de Cross-Validation ---
N_FOLDS = 5

# Obtener lista única de deportistas y asignar fold
deportistas = df_entrenamiento.select("id_unico_deportista").distinct().orderBy("id_unico_deportista").collect()
deportistas_ids = [row["id_unico_deportista"] for row in deportistas]
np.random.seed(42)
np.random.shuffle(deportistas_ids)

# Asignar cada deportista a un fold
fold_assignments = {did: i % N_FOLDS for i, did in enumerate(deportistas_ids)}
fold_map = F.create_map([F.lit(x) for pair in fold_assignments.items() for x in pair])
df_cv = df_entrenamiento.withColumn("fold", fold_map[F.col("id_unico_deportista")])

print(f"Cross-Validation: {N_FOLDS} folds, {len(deportistas_ids)} deportistas")
print("Deportistas por fold:")
df_cv.groupBy("fold").agg(F.countDistinct("id_unico_deportista").alias("n_deportistas"), F.count("*").alias("n_registros")).orderBy("fold").show()

# --- Modelos a evaluar ---
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=columna_objetivo,
    weightCol="weight",
    numTrees=100,
    maxDepth=5,
    seed=42
)
rf.setThresholds([0.5, 0.5])                 # Predice lesión (1) si P(riesgo) >= 0.5

modelos = {"Logistic Regression": lr, "Random Forest": rf}

# --- Evaluadores ---
evaluator_auc_roc = BinaryClassificationEvaluator(
    labelCol=columna_objetivo, rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
evaluator_auc_pr = BinaryClassificationEvaluator(
    labelCol=columna_objetivo, rawPredictionCol="rawPrediction", metricName="areaUnderPR"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol=columna_objetivo, predictionCol="prediction", metricName="fMeasureByLabel", metricLabel=1.0
)
evaluator_precision = MulticlassClassificationEvaluator(
    labelCol=columna_objetivo, predictionCol="prediction", metricName="precisionByLabel", metricLabel=1.0
)
evaluator_recall = MulticlassClassificationEvaluator(
    labelCol=columna_objetivo, predictionCol="prediction", metricName="recallByLabel", metricLabel=1.0
)

# --- Ejecutar Cross-Validation ---
resultados_cv = {nombre: {"AUC-ROC": [], "AUC-PR": [], "F1": [], "Precision": [], "Recall": []} for nombre in modelos}

for fold_i in range(N_FOLDS):
    print(f"\n{'='*50}")
    print(f"FOLD {fold_i + 1}/{N_FOLDS}")
    print(f"{'='*50}")
    
    df_train = df_cv.filter(F.col("fold") != fold_i)
    df_test = df_cv.filter(F.col("fold") == fold_i)
    
    n_train = df_train.count()
    n_test = df_test.count()
    pos_test = df_test.filter(F.col(columna_objetivo) == 1).count()
    print(f"  Train: {n_train} registros | Test: {n_test} registros ({pos_test} positivos)")
    
    for nombre, modelo in modelos.items():
        modelo_fold = modelo.fit(df_train)
        predicciones = modelo_fold.transform(df_test)
        
        auc_roc = evaluator_auc_roc.evaluate(predicciones)
        auc_pr = evaluator_auc_pr.evaluate(predicciones)
        f1 = evaluator_f1.evaluate(predicciones)
        precision = evaluator_precision.evaluate(predicciones)
        recall = evaluator_recall.evaluate(predicciones)
        
        resultados_cv[nombre]["AUC-ROC"].append(auc_roc)
        resultados_cv[nombre]["AUC-PR"].append(auc_pr)
        resultados_cv[nombre]["F1"].append(f1)
        resultados_cv[nombre]["Precision"].append(precision)
        resultados_cv[nombre]["Recall"].append(recall)
        
        print(f"  {nombre}: AUC-ROC={auc_roc:.4f} | AUC-PR={auc_pr:.4f} | F1={f1:.4f}")

print("\n" + "="*50)
print("CROSS-VALIDATION COMPLETADA")
print("="*50)

In [0]:
# --- Tabla resumen de métricas ---
print("\n" + "="*70)
print("RESUMEN DE MÉTRICAS - CROSS-VALIDATION (Media ± Desviación Estándar)")
print("="*70)

resumen_rows = []
for nombre, metricas in resultados_cv.items():
    row = {"Modelo": nombre}
    for metrica, valores in metricas.items():
        media = np.mean(valores)
        std = np.std(valores)
        row[f"{metrica} (media)"] = round(media, 4)
        row[f"{metrica} (std)"] = round(std, 4)
        print(f"  {nombre} | {metrica}: {media:.4f} ± {std:.4f}")
    resumen_rows.append(row)
    print()

df_resumen = pd.DataFrame(resumen_rows)
display(df_resumen)

In [0]:
# --- Gráfico comparativo de métricas por fold ---
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Cross-Validation: Métricas por Fold (GroupKFold por Deportista)", fontsize=14, fontweight='bold')

metricas_nombres = ["AUC-ROC", "AUC-PR", "F1", "Precision", "Recall"]
colores = {"Logistic Regression": "#2196F3", "Random Forest": "#4CAF50"}

for idx, metrica in enumerate(metricas_nombres):
    ax = axes[idx // 3, idx % 3]
    x = np.arange(1, N_FOLDS + 1)
    width = 0.35
    
    for i, (nombre, color) in enumerate(colores.items()):
        valores = resultados_cv[nombre][metrica]
        offset = -width/2 + i * width
        bars = ax.bar(x + offset, valores, width, label=nombre, color=color, alpha=0.8)
        ax.axhline(y=np.mean(valores), color=color, linestyle='--', alpha=0.5, linewidth=1)
    
    ax.set_xlabel("Fold")
    ax.set_ylabel(metrica)
    ax.set_title(metrica)
    ax.set_xticks(x)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

# Ocultar el subplot vacío
axes[1, 2].axis('off')

# Añadir tabla resumen en el subplot vacío
cell_text = []
for nombre in modelos.keys():
    row = [f"{np.mean(resultados_cv[nombre][m]):.3f}±{np.std(resultados_cv[nombre][m]):.3f}" for m in metricas_nombres]
    cell_text.append(row)

table = axes[1, 2].table(
    cellText=cell_text,
    rowLabels=list(modelos.keys()),
    colLabels=metricas_nombres,
    cellLoc='center',
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.2, 1.5)
axes[1, 2].set_title("Resumen (media±std)", fontsize=10)

plt.tight_layout()
plt.show()

In [0]:
# --- DataFrame detallado por fold para análisis posterior ---
rows_detalle = []
for nombre, metricas in resultados_cv.items():
    for fold_i in range(N_FOLDS):
        row = {
            "Modelo": nombre,
            "Fold": fold_i + 1,
            "AUC-ROC": metricas["AUC-ROC"][fold_i],
            "AUC-PR": metricas["AUC-PR"][fold_i],
            "F1": metricas["F1"][fold_i],
            "Precision": metricas["Precision"][fold_i],
            "Recall": metricas["Recall"][fold_i]
        }
        rows_detalle.append(row)

df_detalle_cv = pd.DataFrame(rows_detalle)
display(df_detalle_cv)

# Mejor modelo según AUC-ROC media
mejor_modelo = max(resultados_cv.keys(), key=lambda x: np.mean(resultados_cv[x]["AUC-ROC"]))
print(f"\n✓ Mejor modelo por AUC-ROC media: {mejor_modelo} ({np.mean(resultados_cv[mejor_modelo]['AUC-ROC']):.4f})")

### Cross-Validation con Conjuntos Balanceados

Se llevó a cabo una segunda prueba con cross-validation balanceando los conjuntos de train y test para que contuviesen la misma proporción de entradas con y sin riesgo de lesión.

**Estrategia:** undersampling aleatorio de la clase mayoritaria (negativos / sanos) para igualar la cantidad de la clase minoritaria (positivos / lesionados), tanto en train como en test. Se mantiene la agrupación GroupKFold por deportista para evitar fuga de datos.

**Nota sobre pesos:** en esta versión **no se usa `weightCol`**, ya que el balance se consigue directamente por la composición del dataset. Aplicar los pesos de compensación de la versión desbalanceada sobre un dataset ya balanceado provocaría una doble compensación, forzando al modelo a clasificar casi todo como positivo (Recall≈1.0) sin verdadera capacidad discriminativa.

**Resultados e interpretación:**
- **AUC-ROC** permanece en ~0.53 (igual que en la CV original), lo cual confirma que la capacidad discriminativa real del modelo no ha cambiado.
- **AUC-PR** sube de ~0.04 a ~0.56, pero esto es un artefacto: el baseline de AUC-PR es igual a la proporción de positivos en el test (0.035 en la versión original → 0.50 en la balanceada). Relativo a su baseline, ambas versiones están apenas por encima del azar.
- **Precision** ~0.55 y **Recall** ~0.47 (LR) sobre un test 50/50 implican que el modelo solo supera marginalmente al azar.

**Conclusión:** el problema fundamental no es la estrategia de manejo del desbalance de clases, sino que **las features actuales no contienen suficiente señal predictiva** para separar deportistas en riesgo de los sanos.

#### Limitaciones identificadas

1. **Escasez de datos positivos:** el número real de lesiones es muy reducido, lo que limita la capacidad de cualquier modelo para aprender patrones discriminativos robustos.

2. **Etiquetado semi-artificial de la variable objetivo:** se establece como "en riesgo de lesión" la semana completa previa a cada lesión. Sin embargo, se desconoce si la lesión fue consecuencia de la carga acumulada durante esos días o de una sobrecarga puntual el propio día del evento. Esto introduce ruido en las etiquetas, ya que algunos días marcados como positivos podrían no haber presentado riesgo real, diluyendo la señal que el modelo intenta capturar.

3. **Features sin contexto temporal:** las variables actuales representan el estado de un día aislado, sin capturar la evolución o tendencia de la carga a lo largo del tiempo (aceleración de carga, fatiga acumulada, variabilidad inter-sesión).

4. **Pocos deportistas para GroupKFold:** al agrupar por deportista con un plantel reducido, cada fold contiene muy pocos eventos positivos, generando alta varianza entre folds y estimaciones inestables.

#### Líneas de mejora sugeridas

- **Ingeniería de features temporales:** ratios de carga aguda/crónica (ACWR), tendencias de fatiga con ventanas móviles, deltas día-a-día, interacciones entre variables de carga y recuperación.
- **Redefinición de la variable objetivo:** explorar ventanas más cortas (1-2 días previos) o modelos de supervivencia / time-to-event que no requieran una ventana fija.
- **Validación temporal:** entrenar con temporada 1 y testear con temporada 2 para simular un escenario de uso real (predicción prospectiva).
- **Modelos sensibles al contexto secuencial:** arquitecturas que consuman secuencias de días por deportista (LSTM, Transformer) en lugar de tratar cada registro como independiente.

In [0]:
# --- Cross-Validation con conjuntos BALANCEADOS (undersampling mayoría) ---

print(f"Cross-Validation BALANCEADA: {N_FOLDS} folds, GroupKFold por deportista")
print(f"Estrategia: undersampling de clase negativa para igualar clase positiva")
print("="*70)

# Modelos (misma config que CV original)
lr_bal = LogisticRegression(
    featuresCol="features",
    labelCol=columna_objetivo,
    predictionCol="prediction",
    probabilityCol="probabilidad_vector",
    regParam=0.01,
    threshold=0.5
)

rf_bal = RandomForestClassifier(
    featuresCol="features",
    labelCol=columna_objetivo,
    numTrees=100,
    maxDepth=5,
    seed=42
)
rf_bal.setThresholds([0.5, 0.5])

modelos_bal = {"Logistic Regression": lr_bal, "Random Forest": rf_bal}

# Evaluadores
eval_auc_roc = BinaryClassificationEvaluator(labelCol=columna_objetivo, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
eval_auc_pr = BinaryClassificationEvaluator(labelCol=columna_objetivo, rawPredictionCol="rawPrediction", metricName="areaUnderPR")
eval_f1 = MulticlassClassificationEvaluator(labelCol=columna_objetivo, predictionCol="prediction", metricName="fMeasureByLabel", metricLabel=1.0)
eval_precision = MulticlassClassificationEvaluator(labelCol=columna_objetivo, predictionCol="prediction", metricName="precisionByLabel", metricLabel=1.0)
eval_recall = MulticlassClassificationEvaluator(labelCol=columna_objetivo, predictionCol="prediction", metricName="recallByLabel", metricLabel=1.0)

# Resultados
resultados_cv_bal = {nombre: {"AUC-ROC": [], "AUC-PR": [], "F1": [], "Precision": [], "Recall": []} for nombre in modelos_bal}

for fold_i in range(N_FOLDS):
    print(f"\n{'='*50}")
    print(f"FOLD {fold_i + 1}/{N_FOLDS}")
    print(f"{'='*50}")
    
    df_train_fold = df_cv.filter(F.col("fold") != fold_i)
    df_test_fold = df_cv.filter(F.col("fold") == fold_i)
    
    # --- Balancear TRAIN (undersample negativos) ---
    train_pos = df_train_fold.filter(F.col(columna_objetivo) == 1)
    train_neg = df_train_fold.filter(F.col(columna_objetivo) == 0)
    n_pos_train = train_pos.count()
    n_neg_train = train_neg.count()
    
    fraction_train = min(1.0, n_pos_train / n_neg_train)
    train_neg_sampled = train_neg.sample(withReplacement=False, fraction=fraction_train, seed=42 + fold_i)
    df_train_balanced = train_pos.unionByName(train_neg_sampled)
    
    # --- Balancear TEST (undersample negativos) ---
    test_pos = df_test_fold.filter(F.col(columna_objetivo) == 1)
    test_neg = df_test_fold.filter(F.col(columna_objetivo) == 0)
    n_pos_test = test_pos.count()
    n_neg_test = test_neg.count()
    
    fraction_test = min(1.0, n_pos_test / n_neg_test)
    test_neg_sampled = test_neg.sample(withReplacement=False, fraction=fraction_test, seed=42 + fold_i)
    df_test_balanced = test_pos.unionByName(test_neg_sampled)
    
    n_train_bal = df_train_balanced.count()
    n_test_bal = df_test_balanced.count()
    print(f"  Original  -> Train: {n_pos_train + n_neg_train} ({n_pos_train} pos / {n_neg_train} neg) | Test: {n_pos_test + n_neg_test} ({n_pos_test} pos / {n_neg_test} neg)")
    print(f"  Balanceado-> Train: {n_train_bal} (~50/50) | Test: {n_test_bal} (~50/50)")
    
    for nombre, modelo in modelos_bal.items():
        modelo_fold = modelo.fit(df_train_balanced)
        predicciones = modelo_fold.transform(df_test_balanced)
        
        auc_roc = eval_auc_roc.evaluate(predicciones)
        auc_pr = eval_auc_pr.evaluate(predicciones)
        f1 = eval_f1.evaluate(predicciones)
        precision = eval_precision.evaluate(predicciones)
        recall = eval_recall.evaluate(predicciones)
        
        resultados_cv_bal[nombre]["AUC-ROC"].append(auc_roc)
        resultados_cv_bal[nombre]["AUC-PR"].append(auc_pr)
        resultados_cv_bal[nombre]["F1"].append(f1)
        resultados_cv_bal[nombre]["Precision"].append(precision)
        resultados_cv_bal[nombre]["Recall"].append(recall)
        
        print(f"  {nombre}: AUC-ROC={auc_roc:.4f} | AUC-PR={auc_pr:.4f} | F1={f1:.4f} | Prec={precision:.4f} | Rec={recall:.4f}")

print("\n" + "="*70)
print("CROSS-VALIDATION BALANCEADA COMPLETADA")
print("="*70)

In [0]:
# --- Resumen: Comparativa CV Original vs CV Balanceada ---
print("\n" + "="*70)
print("COMPARATIVA: CV ORIGINAL vs CV BALANCEADA (Media ± Std)")
print("="*70)

rows_comparativa = []
for nombre in modelos_bal.keys():
    for version, res in [("Original", resultados_cv), ("Balanceada", resultados_cv_bal)]:
        row = {"Modelo": nombre, "CV": version}
        for metrica in ["AUC-ROC", "AUC-PR", "F1", "Precision", "Recall"]:
            media = np.mean(res[nombre][metrica])
            std = np.std(res[nombre][metrica])
            row[f"{metrica}"] = f"{media:.4f} ± {std:.4f}"
            row[f"{metrica}_media"] = media
        rows_comparativa.append(row)

df_comparativa = pd.DataFrame(rows_comparativa)
print("\n")
display(df_comparativa[["Modelo", "CV", "AUC-ROC", "AUC-PR", "F1", "Precision", "Recall"]])

# --- Gráfico comparativo ---
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle("Comparativa: CV Original vs CV Balanceada (clase positiva)", fontsize=14, fontweight='bold')

metricas_plot = ["AUC-ROC", "AUC-PR", "F1", "Precision", "Recall"]
colores_version = {"Original": "#90CAF9", "Balanceada": "#FF8A65"}

for idx, metrica in enumerate(metricas_plot):
    ax = axes[idx]
    x = np.arange(len(modelos_bal))
    width = 0.35
    
    for i, (version, res) in enumerate([("Original", resultados_cv), ("Balanceada", resultados_cv_bal)]):
        medias = [np.mean(res[nombre][metrica]) for nombre in modelos_bal.keys()]
        stds = [np.std(res[nombre][metrica]) for nombre in modelos_bal.keys()]
        offset = -width/2 + i * width
        ax.bar(x + offset, medias, width, yerr=stds, label=version,
               color=colores_version[version], alpha=0.85, capsize=4)
    
    ax.set_ylabel(metrica)
    ax.set_title(metrica)
    ax.set_xticks(x)
    ax.set_xticklabels(["LR", "RF"], fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()